<a href="https://colab.research.google.com/github/gauravd12345/language_models/blob/main/seq2seq/seq2seq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [82]:
import re
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split

import nltk
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

dataset = "hf://datasets/PaulineSanchez/Translation_words_and_sentences_english_french/data/train-00000-of-00001-3d14582ea46e1b17.parquet"
df = pd.read_parquet(dataset)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


device: cuda


In [83]:
c1, c2 = df.columns
enc = np.array(df[c1])
dec = np.array(df[c2])

for i in np.random.choice(np.arange(len(df)), 10):
  print(f"{enc[i]:<50} | {dec[i]}")

You've often said so yourself.                     | Tu l'as souvent dit toi-même.
I want to apologize.                               | Je veux présenter mes excuses.
I thought about you the other day.                 | J'ai pensé à vous l'autre jour.
I didn't ask you to come with us.                  | Je ne vous ai pas demandé de venir avec nous.
She bought him a car.                              | Elle lui a acheté une voiture.
Are you sure this is the right train?              | Êtes-vous sûr que c'est le bon train ?
I have an emergency.                               | J’ai une urgence.
Everyone was hurt.                                 | Tout le monde fut blessé.
That isn't exactly what I said.                    | Ce n'est pas exactement ce que j'ai dit.
She finished her homework in an hour.              | Elle a terminé ses devoirs en une heure.


In [84]:
""" setup for encoder dataset """
enc_words = [word_tokenize(seq) for seq in enc]
enc_vocab = set()
for seq in enc_words:
  for word in seq:
    enc_vocab.add(word)

enc_vocab.add("<EOS>")
enc_vocab = sorted(enc_vocab)
enc_vocab_len = len(enc_vocab)

etoi, itoe = {}, {}
for i in range(enc_vocab_len):
    etoi[enc_vocab[i]] = i
    itoe[i] = enc_vocab[i]

print(f"english vocab size: {enc_vocab_len}")
print(enc_vocab[:10])

""" setup for decoder dataset """
dec_words = [word_tokenize(seq) for seq in dec]
dec_vocab = set()
for seq in dec_words:
  for word in seq:
    dec_vocab.add(word)

dec_vocab.add("<EOS>")
dec_vocab = sorted(dec_vocab)
dec_vocab_len = len(dec_vocab)

dtoi, itod = {}, {}
for i in range(dec_vocab_len):
    dtoi[dec_vocab[i]] = i
    itod[i] = dec_vocab[i]

print(f"\nfrench vocab size: {enc_vocab_len}")
print(dec_vocab[:10])

english vocab size: 16895
['!', '$', '%', '&', "'", "''", "'anticonstitutionnellement", "'bribery", "'d", "'em"]

french vocab size: 16895
['!', '$', '%', '&', "'", "''", "'Spot", "'achète", "'maison", "'oublie"]


In [85]:
""" Hyperparameters """

embedding_dim = 300
hidden_size = 300 # sequence length of the encoder output/decoder input vector
max_decoder_seq = 200

In [86]:
E = []
for sentence in enc:
  seq = []
  tok = word_tokenize(sentence) + ["<EOS>"]
  for word in tok:
    e_i = etoi[word]
    seq.append(e_i)
  E.append(torch.tensor(seq))

D = []
for sentence in dec:
  seq = []
  tok = word_tokenize(sentence) + ["<EOS>"]
  for word in tok:
    d_i = dtoi[word]
    seq.append(d_i)
  D.append(torch.tensor(seq))

for i in range(10): # tokenized english to french translations
  print(E[i], D[i])

tensor([1440,   19,  285]) tensor([3832,    0,  232])
tensor([2414,    0,  285]) tensor([942,   0, 232])
tensor([2414,    0,  285]) tensor([939,   0, 232])
tensor([3046,  286,  285]) tensor([3436,  233,  232])
tensor([3096,    0,  285]) tensor([30686,  5230,     0,   232])
tensor([1187,    0,  285]) tensor([  528, 14028,     0,   232])
tensor([1428,    0,  285]) tensor([30684, 17160,     0,   232])
tensor([1611,   19,  285]) tensor([3843,   15,  232])
tensor([2667,    0,  285]) tensor([30686, 27911,     0,   232])
tensor([2667,    0,  285]) tensor([4012,    0,  232])


In [87]:
class Seq2Seq(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc_embed = nn.Embedding(enc_vocab_len, embedding_dim)
        self.dec_embed = nn.Embedding(dec_vocab_len, embedding_dim)
        """ encoder RNN """
        self.E_x = nn.Linear(embedding_dim, hidden_size)
        self.E_h = nn.Linear(hidden_size, hidden_size)
        self.E_y = nn.Linear(hidden_size, hidden_size)

        """ decoder RNN """
        self.D_x = nn.Linear(embedding_dim, hidden_size)
        self.D_h = nn.Linear(hidden_size, hidden_size)
        self.D_y = nn.Linear(hidden_size, dec_vocab_len)

    def forward(self, w, h):
        # pseudocode assuming w is a single word instead of a tensor
        w = self.enc_embed(w)
        for word in w[:-1]:
          h = torch.tanh(self.E_x(word) + self.E_h(h))
          y = self.E_y(h)

        out = []
        eos_tok = dtoi["<EOS>"]

        w = self.dec_embed(torch.tensor(eos_tok))
        h = y
        for _ in range(max_decoder_seq):
          h = torch.tanh(self.D_x(w) + self.D_h(h))
          prob = torch.softmax(self.D_y(h), dim=1)
          pred = prob.argmax().item()

          if pred == eos_tok:
            break

          out.append(pred)
          w = self.dec_embed(torch.tensor(pred))

        return out, h

seq2seq = Seq2Seq().to(device)